# Gene Ontology analysis: retrieval-augmented generation

**Project 4.** Can a small retrieval step over the Gene Ontology make a language model's answers checkable, and does it agree with a standard enrichment analysis on a human gene list?

| Brief | |
|---|---|
| Question | Does retrieving the five closest Gene Ontology terms help a language model answer biology questions with verifiable identifiers, and how does the retrieval agree with an enrichment analysis? |
| Data | `go-basic.obo` (the ontology) and `HUMAN-uniprot.gaf.gz` (human gene annotations), both from the Gene Ontology website. The Gene Ontology site footer states the licence as Creative Commons Attribution 4.0. |
| Models | Embedding: `sentence-transformers/all-MiniLM-L6-v2`. Language model, one swappable function: Claude (application programming interface) or a small open model run locally with PyTorch (`Qwen/Qwen2.5-0.5B-Instruct`). |
| Metrics | Retrieval: how often the known correct term is among the five results. Answers: citation check (are all cited identifiers among the retrieved ones, do they exist). Gene list: overlap with the enrichment answer key, mean and standard deviation over seeds 0, 1 and 42. |
| Time budget | About one day. Everything is cached, so **Run All** is fast after the first run. |

**What is retrieval and what is the language model?** Retrieval (Steps 4 to 7 and 9.1) uses no language model: it only embeds text and compares numbers. The language model (Steps 8 and 9.2) only writes the answer from the terms it is given. Every result below is labelled with the part it belongs to.

**Facts checked against sources while writing this notebook**
- The human annotation file moved in the Gene Ontology release of 2026-06-19: the new file is `HUMAN-uniprot.gaf.gz` in the `annotations/gaf/` folder of `current.geneontology.org`; the older name `goa_human.gaf.gz` stops working after September 2026 (announcement on the Alliance of Genome Resources community forum, confirmed by the Gene Ontology downloads table).
- The annotation file is in Gene Annotation Format 2.2: 17 tab-separated columns, header lines start with `!`, column 3 is the gene symbol, column 4 the relation (may start with `NOT`), column 5 the term identifier (Gene Ontology guide for the format).
- `go-basic.obo` is the version filtered so that the ontology has no cycles, so annotations can be propagated up the graph (Gene Ontology download page).
- Claude model identifiers `claude-sonnet-5-5` and `claude-haiku-5-5` are the ones listed on the current model overview page of the Claude documentation. Older cached copies of that documentation list other names, so if a call fails with "model not found", check that page.
- The embedding size, the text length limit and the number of parameters of the local model are **printed by the packages** in the steps below, not written from memory.
- The correct term for every test question is looked up by its exact **name** in the file you downloaded, and the notebook prints the identifier it found, so you can check each one.

---

## Step 1. Install, imports, versions and settings

**What and why:** one cell installs the packages, one cell holds every import, one cell prints the versions and **writes `requirements.txt` with the exact versions this run used**, and one cell holds all settings.
**Expected output:** package versions as plain lines, then the line `requirements.txt now holds the exact versions listed above.`
**Key terms:** a **kernel** is the Python process that runs the notebook; a **seed** is a number that fixes random choices so a run can be repeated.

In [ ]:
# What: installs the packages in requirements.txt. Run once; restart the kernel afterwards if asked.
# Expected output: pip messages without an error.
%pip install -r requirements.txt

In [ ]:
# What: every import of the notebook, in one place.
# Expected output: no message.
import collections
import getpass
import gzip
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import sys
import urllib.request

import anthropic
import numpy
import pandas
import scipy.stats
import torch
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# What: prints the versions and writes them to requirements.txt (the local build tag after "+" is dropped).
# Expected output: one line per package, then a confirmation line.
PACKAGE_NAMES = ["torch", "numpy", "pandas", "scipy", "sentence-transformers",
                 "transformers", "huggingface_hub", "anthropic"]

print("Python:", sys.version.split()[0])
print("Operating system:", platform.platform())
requirement_lines = []
for package_name in PACKAGE_NAMES:
    package_version = importlib.metadata.version(package_name).split("+")[0]
    print(package_name + ":", package_version)
    requirement_lines.append(package_name + "==" + package_version)

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(requirement_lines) + "\n")
print("requirements.txt now holds the exact versions listed above.")

In [ ]:
# What: all settings in one place, the folders, and two small helper functions.
# Expected output: the computing device and the folder names.
DATA_FOLDER = "data"
RESULTS_FOLDER = "results"
os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

ONTOLOGY_PATH = os.path.join(DATA_FOLDER, "go-basic.obo")
ANNOTATION_PATH = os.path.join(DATA_FOLDER, "HUMAN-uniprot.gaf.gz")
ONTOLOGY_ADDRESSES = [
    "https://purl.obolibrary.org/obo/go/go-basic.obo",
    "https://current.geneontology.org/ontology/go-basic.obo",
]
ANNOTATION_ADDRESSES = [
    "https://current.geneontology.org/annotations/gaf/HUMAN-uniprot.gaf.gz",
]
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LOCAL_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
CLAUDE_MODEL_NAME = "claude-sonnet-5-5"          # a cheaper choice is "claude-haiku-5-5"

BACKENDS_TO_USE = ["claude", "local"]            # remove one word to use only one backend
HOW_MANY_TERMS = 5                               # terms retrieved for every question
MAXIMUM_NEW_TOKENS_LOCAL = 300
TERM_LIMIT = None                                # for a quick trial use 3000 (searches only those terms)
RANDOM_SEEDS = [0, 1, 42]
ANSWER_CACHE_PATH = os.path.join(DATA_FOLDER, "answer_cache.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Computing on:", device)
print("Folders:", DATA_FOLDER, "and", RESULTS_FOLDER)

summary_lines = []                               # every key result is also saved to results/summary.txt


# HELPER FUNCTION
def report(text):
    # Prints a result line and keeps it for the summary file.
    print(text)
    summary_lines.append(text)


# HELPER FUNCTION
def set_seed(seed_value):
    # Fixes all random number generators so that a run can be repeated.
    random.seed(seed_value)
    numpy.random.seed(seed_value)
    torch.manual_seed(seed_value)

## Step 2. Download the data and prove access

**What and why:** downloads the two files (skipped if they are already in `data/`), then checks that the two model sites can be reached and prints what the files contain. The Gene Ontology site asks users to record the **ontology version** and the **date in the annotation file header**, so both are printed here.
**Expected output:** one line per item with `downloaded` or `already on disk` and a size in megabytes; the ontology header; the annotation header and two data lines; the number of columns per line (the format guide says **17**).
**If a line says `FAILED`:** download the file in your browser from https://geneontology.org/docs/download-ontology/ or https://geneontology.org/docs/download-go-annotations/downloads/ (human, UniProt-centric file) and drop it into the `data` folder in the Explorer panel.

In [ ]:
# What: downloads the two files if they are missing and checks the two model sites.
# Expected output: four result lines. Nothing should say FAILED.
# HELPER FUNCTION
def download_if_missing(path, addresses):
    # Returns (status, detail). Tries each address in order.
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return "already on disk", str(round(os.path.getsize(path) / 1e6, 1)) + " megabytes"
    errors = []
    for address in addresses:
        try:
            urllib.request.urlretrieve(address, path)
            return "downloaded", address + " | " + str(round(os.path.getsize(path) / 1e6, 1)) + " megabytes"
        except Exception as error:
            errors.append(address + " -> " + type(error).__name__ + ": " + str(error)[:120])
            if os.path.exists(path):
                os.remove(path)
    return "FAILED", " || ".join(errors)


for item, path, addresses in [("go-basic.obo", ONTOLOGY_PATH, ONTOLOGY_ADDRESSES),
                              ("HUMAN-uniprot.gaf.gz", ANNOTATION_PATH, ANNOTATION_ADDRESSES)]:
    status, detail = download_if_missing(path, addresses)
    report(item + ": " + status + " | " + detail)

for model_name in [EMBEDDING_MODEL_NAME, LOCAL_MODEL_NAME]:
    try:
        hf_hub_download(repo_id=model_name, filename="config.json")
        report(model_name + ": reachable")
    except Exception as error:
        report(model_name + ": FAILED | " + type(error).__name__ + ": " + str(error)[:150])

for path in [ONTOLOGY_PATH, ANNOTATION_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError("Missing file: " + path + ". Download it by hand (see the text above) into the data folder.")

In [ ]:
# What: looks inside both files (version, date and format).
# Expected output: the ontology header, the annotation header, two data lines, and a column count.
print("go-basic.obo, first header lines:")
with open(ONTOLOGY_PATH, encoding="utf-8") as file:
    for line_number, line in enumerate(file):
        if line.startswith("format-version") or line.startswith("data-version") or line.startswith("ontology"):
            report("  " + line.rstrip())
        if line_number > 12:
            break

print("HUMAN-uniprot.gaf.gz, header lines and first data lines:")
shown_data_lines = 0
with gzip.open(ANNOTATION_PATH, "rt", encoding="utf-8") as file:
    for line in file:
        if line.startswith("!gaf-version") or line.startswith("!date-generated") or line.startswith("!generated-by") or line.startswith("!go-version"):
            report("  " + line.rstrip())
        elif not line.startswith("!"):
            columns = line.rstrip("\n").split("\t")
            print("  columns in this line:", len(columns), "| gene symbol:", columns[2], "| relation:", columns[3],
                  "| term identifier:", columns[4], "| evidence:", columns[6])
            shown_data_lines += 1
            if shown_data_lines >= 2:
                break

## Step 3. Basic data checks

**What and why:** reads the ontology into a table and the annotations into a gene-to-term dictionary, and records what is found. Lean checks only.
1. Obsolete terms are removed (the ontology marks them `is_obsolete: true`).
2. Every term keeps its identifier, name, branch (**namespace**), definition and **parent terms** (`is_a` and `relationship: part_of` lines). The parents are needed for the enrichment analysis in Step 10.
3. In the annotation file, lines whose relation contains `NOT` are skipped (they say a gene does **not** have the function), and lines whose term is not in the ontology table (obsolete or alternative identifiers) are skipped and counted.
4. Each annotation is **propagated upward**: a gene linked to a specific term also counts for every ancestor term. `go-basic.obo` is built so that this is safe.

**Expected output:** counts only. Whatever looks odd (many skipped lines, terms without definition) is recorded here and goes into the notes.

In [ ]:
# What: reads the ontology; keeps identifier, name, namespace, definition and parents; drops obsolete terms.
# Expected output: number of terms read, kept, per branch, and a few data quality counts.
def read_ontology_terms(path):
    terms = []
    current = None
    with open(path, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line == "[Term]":
                if current is not None:
                    terms.append(current)
                current = {"identifier": "", "name": "", "namespace": "", "definition": "",
                           "obsolete": False, "parent_identifiers": []}
            elif line.startswith("[") and line.endswith("]"):
                # another kind of block (for example [Typedef]); finish the current term
                if current is not None:
                    terms.append(current)
                current = None
            elif current is not None:
                if line.startswith("id: "):
                    current["identifier"] = line[4:]
                elif line.startswith("name: "):
                    current["name"] = line[6:]
                elif line.startswith("namespace: "):
                    current["namespace"] = line[11:]
                elif line.startswith("def: "):
                    # format:  def: "the definition text." [reference list]
                    first_quote = line.find('"')
                    last_quote = line.rfind('"')
                    current["definition"] = line[first_quote + 1:last_quote]
                elif line.startswith("is_obsolete: true"):
                    current["obsolete"] = True
                elif line.startswith("is_a: "):
                    current["parent_identifiers"].append(line.split()[1])
                elif line.startswith("relationship: part_of "):
                    current["parent_identifiers"].append(line.split()[2])
    if current is not None:
        terms.append(current)
    return pandas.DataFrame(terms)


all_terms = read_ontology_terms(ONTOLOGY_PATH)
terms = all_terms[~all_terms["obsolete"]].reset_index(drop=True)
terms["text"] = terms["name"] + ". " + terms["definition"]

report("Terms read: " + str(len(all_terms)) + " | obsolete terms removed: " + str(int(all_terms["obsolete"].sum())) + " | terms kept: " + str(len(terms)))
for namespace_name, count in terms["namespace"].value_counts().items():
    report("  terms in " + namespace_name + ": " + str(count))
report("Terms without a definition: " + str(int((terms["definition"] == "").sum())))
report("Terms with no parent term: " + str(int(terms["parent_identifiers"].apply(len).eq(0).sum())))
report("Terms whose name appears more than once: " + str(int(terms["name"].duplicated(keep=False).sum())))
report("Terms whose definition is identical to another term's definition: " + str(int((terms["definition"].duplicated(keep=False) & (terms["definition"] != "")).sum())))

In [ ]:
# What: lookup tables and a function that lists every ancestor of a term.
# Expected output: the number of ancestors of one example term.
term_by_identifier = {row["identifier"]: row for row in terms.to_dict("records")}
parent_identifiers = {identifier: row["parent_identifiers"] for identifier, row in term_by_identifier.items()}
ancestor_cache = {}


# HELPER FUNCTION
def ancestors_of(identifier):
    # All terms above this term, following the parent links upward (the term itself is not included).
    if identifier in ancestor_cache:
        return ancestor_cache[identifier]
    found = set()
    to_visit = list(parent_identifiers.get(identifier, []))
    while to_visit:
        current = to_visit.pop()
        if current in found:
            continue
        found.add(current)
        to_visit.extend(parent_identifiers.get(current, []))
    ancestor_cache[identifier] = found
    return found


# HELPER FUNCTION
def find_identifier_by_name(exact_name):
    # Looks up a term by its exact name; returns None if there is no such term.
    matches = terms[terms["name"] == exact_name]
    if len(matches) == 0:
        return None
    biological_matches = matches[matches["namespace"] == "biological_process"]
    return (biological_matches if len(biological_matches) > 0 else matches).iloc[0]["identifier"]


example_identifier = find_identifier_by_name("autophagy")
print("Example term:", example_identifier, "|", term_by_identifier[example_identifier]["name"] if example_identifier else "not found")
if example_identifier:
    print("Number of ancestors of the example term:", len(ancestors_of(example_identifier)))

In [ ]:
# What: reads the human annotation file and propagates every annotation to the ancestor terms.
# Expected output: counts of genes, terms and skipped lines, and the taxon values found in column 13.
gene_to_terms = collections.defaultdict(set)
lines_read = 0
skipped_negative = 0
skipped_unknown_term = 0
taxon_counts = collections.Counter()

with gzip.open(ANNOTATION_PATH, "rt", encoding="utf-8") as file:
    for line in file:
        if line.startswith("!"):
            continue
        columns = line.rstrip("\n").split("\t")
        lines_read += 1
        taxon_counts[columns[12]] += 1
        if "NOT" in columns[3]:
            skipped_negative += 1
            continue
        term_identifier = columns[4]
        if term_identifier not in term_by_identifier:
            skipped_unknown_term += 1
            continue
        gene_symbol = columns[2]
        gene_to_terms[gene_symbol].add(term_identifier)
        gene_to_terms[gene_symbol].update(ancestors_of(term_identifier))

term_to_genes = collections.defaultdict(set)
for gene_symbol, term_set in gene_to_terms.items():
    for term_identifier in term_set:
        term_to_genes[term_identifier].add(gene_symbol)

report("Annotation lines read: " + str(lines_read))
report("Skipped, negative relation (NOT): " + str(skipped_negative) + " | skipped, term not in the ontology table: " + str(skipped_unknown_term))
report("Genes with at least one annotation (used as the background): " + str(len(gene_to_terms)))
report("Terms with at least one gene after propagation: " + str(len(term_to_genes)))
report("Most common values in the taxon column: " + str(taxon_counts.most_common(3)))

## Step 4. Data into PyTorch: one embedding per term (cached)

**What and why:** the embedding model turns each term's text (name and definition) into a list of numbers, an **embedding**. Texts with similar meaning get similar numbers. The result is one PyTorch table with one row per term. The first run downloads the model and takes roughly 5 to 10 minutes on an ordinary processor; later runs load the cache. The cache also stores the identifiers, so a changed ontology file can never be paired with an old cache.
**Expected output:** the embedding size and the text length limit (both printed by the package), the number of terms that are longer than the limit and therefore cut off, the shape of one item and of the whole table, and a row length of 1.0 (all rows are normalised, so closeness is one matrix multiplication).

In [ ]:
# What: loads the embedding model, embeds every term once, and caches the result on disk.
# Expected output: the printed facts, a progress bar on the first run, then the table shapes.
if TERM_LIMIT is None:
    searchable_terms = terms
    EMBEDDING_PATH = os.path.join(DATA_FOLDER, "go_term_embeddings.npz")
else:
    searchable_terms = terms.iloc[:TERM_LIMIT].reset_index(drop=True)
    EMBEDDING_PATH = os.path.join(DATA_FOLDER, "go_term_embeddings_trial.npz")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
report("Embedding size (printed by the package): " + str(embedding_model.get_sentence_embedding_dimension()))
report("Text length limit in word pieces (printed by the package): " + str(embedding_model.max_seq_length))

token_counts = [len(ids) for ids in embedding_model.tokenizer(searchable_terms["text"].tolist())["input_ids"]]
report("Terms with a text longer than the limit (the end is cut off): " + str(sum(count > embedding_model.max_seq_length for count in token_counts)))

searchable_identifiers = searchable_terms["identifier"].tolist()
cache_is_valid = False
if os.path.exists(EMBEDDING_PATH):
    saved = numpy.load(EMBEDDING_PATH)
    if saved["identifiers"].tolist() == searchable_identifiers:
        term_embeddings_array = saved["embeddings"]
        cache_is_valid = True
        print("Loaded the cached embeddings.")
    else:
        print("The cache belongs to a different ontology file; computing again.")

if not cache_is_valid:
    term_embeddings_array = embedding_model.encode(
        searchable_terms["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True
    )
    numpy.savez(EMBEDDING_PATH, embeddings=term_embeddings_array, identifiers=numpy.array(searchable_identifiers))

term_embeddings = torch.from_numpy(term_embeddings_array).float()      # one row per term
print("Shape of one item:", tuple(term_embeddings[0].shape))
print("Shape of the whole table (terms, numbers per term):", tuple(term_embeddings.shape))
print("Length of the first row (should be 1.0):", round(float(numpy.linalg.norm(term_embeddings_array[0])), 4))

## Step 5. Retrieval: the five closest terms

**What and why:** the question is embedded the same way, and one matrix multiplication gives its closeness (**cosine similarity**) to every term. `torch.topk` returns the five largest. The optional `namespace` setting limits the search to one branch (`biological_process`, `molecular_function` or `cellular_component`).
**Expected output (further down):** the number of numbers that are *learned* in this step (none: the embedding model is frozen) and the shape of one search result.

In [ ]:
# What: the search function. PyTorch does the similarity calculation.
# Expected output: no message.
namespace_of_each_term = searchable_terms["namespace"].to_numpy()


# HELPER FUNCTION
def search_terms(question, how_many=HOW_MANY_TERMS, namespace=None):
    question_embedding = torch.from_numpy(
        embedding_model.encode([question], normalize_embeddings=True)[0]
    ).float()
    similarities = term_embeddings @ question_embedding                    # one number per term
    if namespace is not None:
        outside_namespace = torch.from_numpy(namespace_of_each_term != namespace)
        similarities = similarities.masked_fill(outside_namespace, -1.0)   # -1 is the lowest possible score
    best_values, best_positions = torch.topk(similarities, k=how_many)
    found = searchable_terms.iloc[best_positions.tolist()][["identifier", "name", "definition"]].copy()
    found["similarity"] = best_values.tolist()
    return found

In [ ]:
# What: checks the search function and takes a first look at three questions.
# Expected output: the shape check, then three terms for each question.
frozen_numbers = sum(parameter.numel() for parameter in embedding_model.parameters())
print("Learnable numbers in the retrieval step: 0 | frozen numbers in the embedding model:", frozen_numbers)
check_result = search_terms("how cells recycle their own components")
print("Shape of one search result (rows, columns):", check_result.shape)

for question in ["ageing of cells", "immune response against bacteria", "chromatin remodelling"]:
    print("QUESTION:", question)
    found = search_terms(question, how_many=3)
    for _, row in found.iterrows():
        print("  " + row["identifier"] + " | " + row["name"] + " | similarity " + str(round(row["similarity"], 3)))

## Step 6. Baselines for retrieval

**What and why:** a score means little without something to compare it with. Two do-nothing baselines:
1. **Chance:** a random choice of five terms. The chance that one named term is among them is five divided by the number of terms.
2. **Naive word search:** counts how many words of the question appear in a term's text. It knows nothing about meaning and cannot match different spellings.

**Expected output:** the chance level, and a check that the naive search returns five terms.

In [ ]:
# What: defines the naive word search and prints the chance level.
# Expected output: two lines.
# HELPER FUNCTION
def naive_word_search(question, how_many=HOW_MANY_TERMS):
    # Counts how many question words appear in the term text. No meaning, only exact words.
    question_words = set(question.lower().replace("?", "").split())
    counts = searchable_terms["text"].str.lower().apply(lambda text: sum(word in text for word in question_words))
    best_positions = numpy.argsort(-counts.to_numpy(), kind="stable")[:how_many]
    return searchable_terms.iloc[best_positions]


report("Chance that one named term is among five random terms: " + str(round(HOW_MANY_TERMS / len(searchable_terms), 5)))
print("Naive word search returns", len(naive_word_search("how cells recycle their own components")), "terms.")

## Step 7. The language model part: one swappable function and a citation check

**What and why:** `generate_answer(prompt, backend)` is the only place that knows which language model is used. Everything else calls this function, so a third backend can be added later without changing any other cell.

| Backend | Quality | Cost | Speed |
|---|---|---|---|
| `"claude"` | Best | A few cents for the whole notebook | Fast |
| `"local"` | Weaker, often ignores the rules | Free | Slow on a processor; the first call downloads the model |

Answers are saved in `data/answer_cache.json`, so a re-run costs nothing and returns the same text. (Without the cache a language model answers a little differently every time.)

**The citation check.** Rule: every identifier in the answer must be one of the five retrieved identifiers. The check also records whether each cited identifier exists in the ontology at all. An identifier that does not exist is invented.

**Your Claude key** is typed into a hidden box, kept only in the memory of the notebook, and never written to a file. If the environment variable `ANTHROPIC_API_KEY` is set it is used without asking. Press Enter on an empty box to use only the local backend.
**Expected output:** the backends that work, one test answer per backend, and a demonstration of the citation check on a made-up answer.

In [ ]:
# What: loads the answer cache and asks for the Claude key.
# Expected output: the number of cached answers and the backends that will be used.
if os.path.exists(ANSWER_CACHE_PATH):
    with open(ANSWER_CACHE_PATH, encoding="utf-8") as file:
        answer_cache = json.load(file)
else:
    answer_cache = {}
print("Answers already cached:", len(answer_cache))

claude_key = os.environ.get("ANTHROPIC_API_KEY", "")
if claude_key == "" and "claude" in BACKENDS_TO_USE:
    claude_key = getpass.getpass("Claude key (press Enter to skip the Claude backend): ").strip()
if claude_key == "":
    BACKENDS_TO_USE = [backend for backend in BACKENDS_TO_USE if backend != "claude"]
print("Backends that will be used:", BACKENDS_TO_USE)

In [ ]:
# What: the two backends and the one swappable function.
# Expected output: no message.
local_tokenizer = None
local_model = None


# HELPER FUNCTION
def generate_with_claude(prompt):
    client = anthropic.Anthropic(api_key=claude_key)
    response = client.messages.create(
        model=CLAUDE_MODEL_NAME,
        max_tokens=600,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


# HELPER FUNCTION
def generate_with_local_model(prompt):
    global local_tokenizer, local_model
    if local_model is None:                                  # load only once
        local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
        local_model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_NAME).to(device)
        local_model.eval()
        print("Numbers in the local model (printed by the package):", sum(parameter.numel() for parameter in local_model.parameters()))
    chat_text = local_tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = local_tokenizer(chat_text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = local_model.generate(**inputs, max_new_tokens=MAXIMUM_NEW_TOKENS_LOCAL, do_sample=False)
    new_pieces = output[0][inputs["input_ids"].shape[1]:]
    return local_tokenizer.decode(new_pieces, skip_special_tokens=True).strip()


# HELPER FUNCTION
def generate_answer(prompt, backend):
    # THE swappable function: the same call for every backend, with a cache on disk.
    model_name = CLAUDE_MODEL_NAME if backend == "claude" else LOCAL_MODEL_NAME
    cache_key = backend + "||" + model_name + "||" + prompt
    if cache_key in answer_cache:
        return answer_cache[cache_key]
    if backend == "claude":
        answer = generate_with_claude(prompt)
    elif backend == "local":
        answer = generate_with_local_model(prompt)
    else:
        raise ValueError("Unknown backend: " + backend)
    answer_cache[cache_key] = answer
    with open(ANSWER_CACHE_PATH, "w", encoding="utf-8") as file:
        json.dump(answer_cache, file)
    return answer

In [ ]:
# What: one tiny test call for each backend, to prove that it works.
# Expected output: one line per backend. A failing backend is removed from the list.
for backend in list(BACKENDS_TO_USE):
    try:
        print(backend, "->", generate_answer("Reply with the single word: ready", backend))
    except Exception as error:
        print(backend, "FAILED:", type(error).__name__, str(error)[:300])
        BACKENDS_TO_USE = [name for name in BACKENDS_TO_USE if name != backend]
report("Backends that work: " + str(BACKENDS_TO_USE))

In [ ]:
# What: the two prompt builders and the citation checker.
# The two prompts are identical except for one thing: whether the retrieved terms are pasted in.
# Expected output: a demonstration of the citation check on a made-up answer.
# HELPER FUNCTION
def build_prompt_with_retrieval(question, found):
    lines = []
    for _, row in found.iterrows():
        lines.append("- " + row["identifier"] + " | " + row["name"] + ": " + row["definition"])
    return (
        "You are helping a biology student.\n"
        "Answer the question below using ONLY the Gene Ontology terms listed. "
        "After every statement, cite the identifier of the term it comes from, written exactly like GO:0000000. "
        "Do not mention any identifier that is not in the list. "
        "If the listed terms are not enough to answer, say so clearly. "
        "Keep the answer under 120 words.\n\n"
        "Question: " + question + "\n\n"
        "Gene Ontology terms:\n" + "\n".join(lines) + "\n"
    )


# HELPER FUNCTION
def build_prompt_without_retrieval(question):
    return (
        "You are helping a biology student.\n"
        "Answer the question below from your own knowledge. "
        "After every statement, cite the identifier of the Gene Ontology term it relates to, "
        "written exactly like GO:0000000. "
        "If you are not sure of an identifier, say so instead of guessing. "
        "Keep the answer under 120 words.\n\n"
        "Question: " + question + "\n"
    )


IDENTIFIER_PATTERN = re.compile(r"GO:\d{7}")        # how every identifier is written


# HELPER FUNCTION
def check_citations(answer, retrieved_identifiers):
    cited = sorted(set(IDENTIFIER_PATTERN.findall(answer)))
    inside_retrieved = [identifier for identifier in cited if identifier in retrieved_identifiers]
    outside_retrieved = [identifier for identifier in cited if identifier not in retrieved_identifiers]
    existing = [identifier for identifier in cited if identifier in term_by_identifier]
    not_existing = [identifier for identifier in cited if identifier not in term_by_identifier]
    return {
        "cited": cited,
        "inside_retrieved": inside_retrieved,
        "outside_retrieved": outside_retrieved,
        "existing_in_ontology": existing,
        "not_in_ontology": not_existing,
        "passes": len(cited) > 0 and len(outside_retrieved) == 0,
    }


demonstration_found = search_terms("How do cells recycle their own components?")
demonstration = check_citations(
    "One valid [" + demonstration_found.iloc[0]["identifier"] + "], one outside the list GO:0000001, one that does not exist GO:9999999.",
    demonstration_found["identifier"].tolist(),
)
print("Demonstration of the check on a made-up answer:", demonstration)

In [ ]:
# What: a first run on one example question, for every working backend, with and without retrieval.
# Expected output: the retrieved terms, then each answer and its citation check.
example_question = "Which biological processes link chromatin structure to gene activity?"
example_found = search_terms(example_question)
print("QUESTION:", example_question)
for _, row in example_found.iterrows():
    print("  retrieved: " + row["identifier"] + " | " + row["name"])
for backend in BACKENDS_TO_USE:
    for condition in ["with retrieval", "without retrieval"]:
        prompt = build_prompt_with_retrieval(example_question, example_found) if condition == "with retrieval" else build_prompt_without_retrieval(example_question)
        answer = generate_answer(prompt, backend)
        checked = check_citations(answer, example_found["identifier"].tolist())
        print()
        print("[" + backend + ", " + condition + "] " + answer)
        print("  cited:", checked["cited"], "| not in the ontology:", checked["not_in_ontology"], "| outside the retrieved list:", checked["outside_retrieved"], "| passes:", checked["passes"])

## Step 8. Test once: retrieval quality and the effect of retrieval on answers

The question set below was written **before** looking at any result and is not changed afterwards. Each question has one known correct term, given by its exact name; the identifier is looked up in your file and printed so you can check it. Some questions avoid the words of the term name, and one uses British spelling on purpose.

### 8.1 Retrieval only (no language model)
A question is a **hit** if the correct term is among the five results. A **related hit** also accepts a result that is an ancestor or a descendant of the correct term (a looser measure). The embedding search is compared with the two baselines from Step 6.
**Expected output:** the question list with the identifier found, one line per question, and three summary lines.

In [ ]:
# What: the fixed question set (question, exact name of the correct term) and the identifier lookup.
# Expected output: one line per question showing the identifier found. A question whose term name is not found is skipped.
test_questions = [
    ("How do cells digest and recycle their own components?", "autophagy"),
    ("How do cells repair damaged genetic material?", "DNA repair"),
    ("What is the process of programmed cell death?", "apoptotic process"),
    ("How do cells get older?", "aging"),
    ("How do cells respond to oxidative stress?", "response to oxidative stress"),
    ("How is insulin released from pancreatic cells?", "insulin secretion"),
    ("How do neurons pass signals across a synapse?", "chemical synaptic transmission"),
    ("chromatin remodelling", "chromatin remodeling"),                  # British spelling in the question, American in the ontology
    ("How does the body defend itself against bacteria?", "defense response to bacterium"),
    ("How is a methyl group added to genetic material?", "DNA methylation"),
]

known_questions = []
for question, correct_name in test_questions:
    correct_identifier = find_identifier_by_name(correct_name)
    if correct_identifier is None:
        print("SKIPPED (name not found in the ontology file):", correct_name)
        continue
    known_questions.append((question, correct_name, correct_identifier))
    print(correct_identifier + " | " + correct_name + "  <-  " + question)
report("Questions with a known correct term: " + str(len(known_questions)) + " of " + str(len(test_questions)))


# HELPER FUNCTION
def terms_are_related(first_identifier, second_identifier):
    # True if the two terms are the same, or one is an ancestor of the other.
    return (first_identifier == second_identifier
            or second_identifier in ancestors_of(first_identifier)
            or first_identifier in ancestors_of(second_identifier))

In [ ]:
# What: measures how often the correct term is among the five results, for the embedding search and the naive search.
# Expected output: one line per question, then three summary lines (embedding, naive word search, chance).
retrieval_rows = []
for question, correct_name, correct_identifier in known_questions:
    embedding_found = search_terms(question)
    naive_found = naive_word_search(question)
    embedding_identifiers = embedding_found["identifier"].tolist()
    naive_identifiers = naive_found["identifier"].tolist()
    row = {
        "question": question, "correct_term": correct_name,
        "embedding_top_one": embedding_identifiers[0] == correct_identifier,
        "embedding_hit": correct_identifier in embedding_identifiers,
        "embedding_related_hit": any(terms_are_related(identifier, correct_identifier) for identifier in embedding_identifiers),
        "naive_hit": correct_identifier in naive_identifiers,
        "naive_related_hit": any(terms_are_related(identifier, correct_identifier) for identifier in naive_identifiers),
    }
    retrieval_rows.append(row)
    print(("HIT   " if row["embedding_hit"] else "miss  ") + correct_name + " | top result: " + embedding_found.iloc[0]["name"])

retrieval_table = pandas.DataFrame(retrieval_rows)
retrieval_table.to_csv(os.path.join(RESULTS_FOLDER, "retrieval_per_question.csv"), index=False)
question_count = len(retrieval_table)
report("Embedding search: correct term first in " + str(int(retrieval_table["embedding_top_one"].sum())) + ", among the five in " + str(int(retrieval_table["embedding_hit"].sum())) + ", related among the five in " + str(int(retrieval_table["embedding_related_hit"].sum())) + " of " + str(question_count) + " questions")
report("Naive word search: correct term among the five in " + str(int(retrieval_table["naive_hit"].sum())) + ", related among the five in " + str(int(retrieval_table["naive_related_hit"].sum())) + " of " + str(question_count) + " questions")
report("Chance level for the correct term among five random terms, per question: " + str(round(HOW_MANY_TERMS / len(searchable_terms), 5)))

### 8.2 Language model: answers with and without retrieval
Each question is answered by each working backend under two conditions. The retrieved set is computed for both conditions, but in the *without retrieval* condition the model never sees it; it only serves as a reference.

| Measure | Meaning |
|---|---|
| cited | how many different identifiers the answer cites |
| existing | how many of them exist in the ontology (the rest are invented) |
| inside retrieved | how many are among the five retrieved terms |
| passes | the strict check: at least one identifier, and every one is among the retrieved terms |
| relevant | at least one cited identifier is the correct term, or an ancestor or descendant of it (identifier looked up in the ontology, not taken from the answer text) |

**Expected output:** summary lines per backend and condition. Answers and their checks are saved in `results/`. A few minutes for Claude; the local model may need 10 to 30 minutes on a processor for the first run.

In [ ]:
# What: runs the comparison for every question, backend and condition.
# Expected output: one summary line per backend and condition.
comparison_rows = []
answer_records = []
for question, correct_name, correct_identifier in known_questions:
    found = search_terms(question)
    retrieved_identifiers = found["identifier"].tolist()
    for backend in BACKENDS_TO_USE:
        for condition in ["with retrieval", "without retrieval"]:
            prompt = build_prompt_with_retrieval(question, found) if condition == "with retrieval" else build_prompt_without_retrieval(question)
            answer = generate_answer(prompt, backend)
            checked = check_citations(answer, retrieved_identifiers)
            relevant = any(terms_are_related(identifier, correct_identifier) for identifier in checked["existing_in_ontology"])
            comparison_rows.append({
                "question": question, "backend": backend, "condition": condition,
                "cited": len(checked["cited"]), "existing": len(checked["existing_in_ontology"]),
                "inside_retrieved": len(checked["inside_retrieved"]),
                "passes": checked["passes"], "relevant": relevant,
            })
            answer_records.append({"question": question, "backend": backend, "condition": condition, "answer": answer, "checked": checked})

comparison_table = pandas.DataFrame(comparison_rows)
comparison_table.to_csv(os.path.join(RESULTS_FOLDER, "comparison_per_question.csv"), index=False)
with open(os.path.join(RESULTS_FOLDER, "answers_with_and_without_retrieval.txt"), "w", encoding="utf-8") as file:
    for record in answer_records:
        file.write("QUESTION: " + record["question"] + " | " + record["backend"] + " | " + record["condition"] + "\n")
        file.write(record["answer"] + "\n")
        file.write("CITATION CHECK: " + json.dumps(record["checked"]) + "\n\n")

for backend in BACKENDS_TO_USE:
    for condition in ["with retrieval", "without retrieval"]:
        part = comparison_table[(comparison_table["backend"] == backend) & (comparison_table["condition"] == condition)]
        report(backend + " | " + condition + " | answers: " + str(len(part))
               + " | identifiers cited: " + str(int(part["cited"].sum()))
               + " | existing: " + str(int(part["existing"].sum()))
               + " | inside retrieved: " + str(int(part["inside_retrieved"].sum()))
               + " | answers passing the strict check: " + str(int(part["passes"].sum()))
               + " | answers with a relevant term: " + str(int(part["relevant"].sum())))

## Step 9. Measuring near-duplicate terms

**What and why:** the ontology has many closely related terms, so the five retrieved terms may be almost the same idea. For every test question this step records the highest similarity between any two of the five retrieved terms (1.0 means identical).
**Expected output:** one line per question and one summary line.

In [ ]:
# What: highest similarity between two of the five retrieved terms, per question.
# Expected output: one line per question, then the mean.
highest_values = []
for question, correct_name, correct_identifier in known_questions:
    found = search_terms(question)
    five_embeddings = term_embeddings[found.index.tolist()]
    pairwise = five_embeddings @ five_embeddings.T
    pairwise.fill_diagonal_(-1.0)                                 # ignore each term compared with itself
    highest_values.append(pairwise.max().item())
    print(str(round(highest_values[-1], 3)) + " | " + question)
report("Mean of the highest similarity between two retrieved terms: " + str(round(float(numpy.mean(highest_values)), 3)))

## Step 10. Gene list test with an enrichment analysis as the answer key

**Idea:** take a list of human genes, ask which Gene Ontology terms are **over-represented** in it (the answer key), then ask the retrieval pipeline the same question from the gene symbols and compare.

**The test** (over-representation analysis): for each term, the hypergeometric test gives the probability of finding at least this many genes of the list among the genes of the term if the list had been drawn at random from all annotated human genes. The probabilities are corrected for testing many terms with the **Benjamini-Hochberg** method (false discovery rate). A term is significant if the corrected value is below 0.05.

**Choices (all are simplifications):**
1. Annotations are propagated upward (Step 3).
2. Only the `biological_process` branch is tested, with terms of 5 to 1,500 genes (very broad terms are always enriched and say little). Terms are tested only if at least 2 list genes belong to them.
3. The background is every human gene with at least one annotation.
4. **The list is planted:** 30 genes come from the genes of one chosen term (`autophagy`) and 10 random genes are added as noise. This is **circular** (the list was built from the same annotation file that the test uses), so it shows that the code works but flatters the enrichment analysis.
5. **Seeds 0, 1 and 42** give three different random lists. The spread between them only reflects which genes were drawn; it is not the full uncertainty.

**Controls:** a random list of 40 genes should give no significant term, and the enrichment test is checked once against an exact counting calculation.
**Expected output (further down):** per seed, the significant terms and the agreement of the retrieval pipeline with the answer key; then the mean and standard deviation over the seeds.

In [ ]:
# What: the enrichment test and the agreement measure.
# Expected output: no message.
ENRICHMENT_NAMESPACE = "biological_process"
MINIMUM_TERM_SIZE = 5
MAXIMUM_TERM_SIZE = 1500
MINIMUM_OVERLAP = 2
ANSWER_KEY_SIZE = 10
SEED_TERM_NAME = "autophagy"
PLANTED_GENE_COUNT = 30
NOISE_GENE_COUNT = 10


# HELPER FUNCTION
def adjust_probability_values(probability_values):
    # Benjamini-Hochberg correction, written out step by step.
    probability_values = numpy.asarray(probability_values, dtype=float)
    count = len(probability_values)
    order = numpy.argsort(probability_values)
    scaled = probability_values[order] * count / numpy.arange(1, count + 1)
    scaled = numpy.minimum.accumulate(scaled[::-1])[::-1]          # make the sequence non-decreasing
    adjusted = numpy.empty(count)
    adjusted[order] = numpy.minimum(scaled, 1.0)
    return adjusted


# HELPER FUNCTION
def run_enrichment(gene_list):
    genes_in_background = [gene for gene in gene_list if gene in gene_to_terms]
    list_size = len(genes_in_background)
    background_size = len(gene_to_terms)
    overlap_counts = collections.Counter()
    for gene in genes_in_background:
        for term_identifier in gene_to_terms[gene]:
            overlap_counts[term_identifier] += 1
    rows = []
    for term_identifier, overlap in overlap_counts.items():
        term = term_by_identifier.get(term_identifier)
        term_size = len(term_to_genes[term_identifier])
        if term is None or term["namespace"] != ENRICHMENT_NAMESPACE:
            continue
        if overlap < MINIMUM_OVERLAP or term_size < MINIMUM_TERM_SIZE or term_size > MAXIMUM_TERM_SIZE:
            continue
        probability_value = scipy.stats.hypergeom.sf(overlap - 1, background_size, term_size, list_size)
        rows.append({"identifier": term_identifier, "name": term["name"], "genes_in_list": overlap,
                     "genes_in_term": term_size, "probability_value": probability_value})
    table = pandas.DataFrame(rows)
    if len(table) == 0:
        return table
    table["adjusted_probability_value"] = adjust_probability_values(table["probability_value"])
    return table.sort_values("adjusted_probability_value").reset_index(drop=True)


# HELPER FUNCTION
def agreement_with_answer_key(identifiers, answer_key_identifiers):
    # Returns (how many identifiers, exact matches, related matches) against the answer key.
    exact = [identifier for identifier in identifiers if identifier in answer_key_identifiers]
    related = [identifier for identifier in identifiers
               if any(terms_are_related(identifier, key) for key in answer_key_identifiers)]
    return len(identifiers), len(exact), len(related)


seed_term_identifier = find_identifier_by_name(SEED_TERM_NAME)
report("Seed term: " + str(seed_term_identifier) + " | " + SEED_TERM_NAME + " | genes annotated to it (after propagation): " + str(len(term_to_genes[seed_term_identifier])))
if not (MINIMUM_TERM_SIZE <= len(term_to_genes[seed_term_identifier]) <= MAXIMUM_TERM_SIZE):
    print("WARNING: the seed term is outside the size limits and will not be tested. Choose another SEED_TERM_NAME.")

In [ ]:
# What: checks the enrichment test once against an exact counting calculation (a control).
# Expected output: one line saying whether the two probability values agree.
control_genes = sorted(term_to_genes[seed_term_identifier])[:PLANTED_GENE_COUNT]
control_result = run_enrichment(control_genes)
control_row = control_result[control_result["identifier"] == seed_term_identifier].iloc[0]
background_size = len(gene_to_terms)
term_size = int(control_row["genes_in_term"])
list_size = len([gene for gene in control_genes if gene in gene_to_terms])
overlap = int(control_row["genes_in_list"])
exact_probability = sum(math.comb(term_size, k) * math.comb(background_size - term_size, list_size - k)
                        for k in range(overlap, min(term_size, list_size) + 1)) / math.comb(background_size, list_size)
report("Control: enrichment probability value " + format(control_row["probability_value"], ".3e")
       + " | exact counting " + format(exact_probability, ".3e")
       + " | agree: " + str(math.isclose(control_row["probability_value"], exact_probability, rel_tol=1e-6)))

In [ ]:
# What: for each seed, plants a gene list, computes the answer key, runs a random control list, and asks the
#       retrieval pipeline (and every working backend) the same question built from the gene symbols.
# Expected output: per seed, the gene list, the top of the answer key, and the agreement lines.
seed_results = []
seed_term_genes = sorted(term_to_genes[seed_term_identifier])
other_genes = sorted(set(gene_to_terms) - set(seed_term_genes))
all_gene_symbols = sorted(gene_to_terms)

for seed_value in RANDOM_SEEDS:
    random_generator = numpy.random.default_rng(seed_value)
    planted_genes = random_generator.choice(seed_term_genes, size=PLANTED_GENE_COUNT, replace=False).tolist()
    noise_genes = random_generator.choice(other_genes, size=NOISE_GENE_COUNT, replace=False).tolist()
    gene_list = sorted(set(planted_genes) | set(noise_genes))

    enrichment_result = run_enrichment(gene_list)
    significant = enrichment_result[enrichment_result["adjusted_probability_value"] < 0.05]
    answer_key = significant.head(ANSWER_KEY_SIZE)
    answer_key_identifiers = answer_key["identifier"].tolist()
    seed_term_rank = (enrichment_result["identifier"].tolist().index(seed_term_identifier) + 1
                      if seed_term_identifier in enrichment_result["identifier"].tolist() else None)

    random_list = random_generator.choice(all_gene_symbols, size=40, replace=False).tolist()
    random_result = run_enrichment(random_list)
    random_significant = int((random_result["adjusted_probability_value"] < 0.05).sum()) if len(random_result) > 0 else 0

    print()
    print("SEED " + str(seed_value) + " | gene list: " + ", ".join(gene_list))
    report("Seed " + str(seed_value) + ": significant terms " + str(len(significant)) + " | rank of the seed term " + str(seed_term_rank) + " | significant terms in a random control list of 40 genes: " + str(random_significant))
    for position, (_, row) in enumerate(answer_key.iterrows(), start=1):
        print("  answer key " + str(position) + ". " + row["identifier"] + " | " + row["name"] + " | genes " + str(row["genes_in_list"]) + " of " + str(row["genes_in_term"]) + " | adjusted probability value " + format(row["adjusted_probability_value"], ".2e"))

    gene_question = "Which biological processes do these human genes have in common? Genes: " + ", ".join(gene_list)
    found_for_genes = search_terms(gene_question, how_many=HOW_MANY_TERMS, namespace=ENRICHMENT_NAMESPACE)
    retrieved_identifiers = found_for_genes["identifier"].tolist()
    count, exact, related = agreement_with_answer_key(retrieved_identifiers, answer_key_identifiers)
    seed_result = {"seed": seed_value, "retrieval_exact": exact, "retrieval_related": related,
                   "significant_terms": len(significant), "random_control_significant": random_significant}
    report("Seed " + str(seed_value) + " | retrieval only (five terms): exact overlap with the answer key " + str(exact) + ", related overlap " + str(related))

    for backend in BACKENDS_TO_USE:
        for condition in ["with retrieval", "without retrieval"]:
            prompt = build_prompt_with_retrieval(gene_question, found_for_genes) if condition == "with retrieval" else build_prompt_without_retrieval(gene_question)
            answer = generate_answer(prompt, backend)
            checked = check_citations(answer, retrieved_identifiers)
            count, exact, related = agreement_with_answer_key(checked["existing_in_ontology"], answer_key_identifiers)
            report("Seed " + str(seed_value) + " | " + backend + ", " + condition + ": identifiers cited " + str(len(checked["cited"]))
                   + ", not in the ontology " + str(len(checked["not_in_ontology"])) + ", passes the citation check " + str(checked["passes"])
                   + ", exact overlap " + str(exact) + ", related overlap " + str(related))
            seed_result[backend + " " + condition + " exact"] = exact
            seed_result[backend + " " + condition + " related"] = related
    seed_results.append(seed_result)

pandas.DataFrame(seed_results).to_csv(os.path.join(RESULTS_FOLDER, "gene_list_per_seed.csv"), index=False)

In [ ]:
# What: mean and standard deviation over the seeds (sample standard deviation, n minus 1).
# Expected output: one line per measure.
seed_table = pandas.DataFrame(seed_results)
for column_name in seed_table.columns:
    if column_name == "seed":
        continue
    values = seed_table[column_name].to_numpy(dtype=float)
    report(column_name + ": mean " + str(round(float(values.mean()), 2)) + ", standard deviation " + str(round(float(values.std(ddof=1)), 2)) + " over seeds " + str(RANDOM_SEEDS))

## Step 11. Conclusions

**Not written yet, on purpose.** The conclusions (results, meaning, limits, next steps) are written only after the outputs of the saved notebook exist and have been read. Until then nothing here claims a result.

**Possible reasons for noisy or misleading results, to check against the outputs (each one is a possible cause, **not tested here** unless a number above shows it):**
1. **General-purpose embedding model.** It was not trained on biology, so it may rank ontology wording and gene symbols poorly. Related numbers: Step 8.1 and the retrieval lines in Step 10.
2. **Near-duplicate terms.** Parent and child terms with similar definitions may fill the five places. Related numbers: Step 3 (identical definitions) and Step 9.
3. **Tiny test set.** Ten questions, written by one person; one more or fewer hit changes every share. The spread over three seeds only reflects which genes were drawn.
4. **Invented facts.** A language model can cite identifiers that do not exist or that belong to another topic. Related numbers: the `existing`, `passes` and `relevant` measures in Step 8.2.
5. **The answer key is itself a choice.** It depends on the background set, the term size limits, the propagation rule and the correction method, and the planted list is circular (Step 10).
6. **Texts cut off by the embedding model.** Step 4 counts them.

**When the outputs exist:** send the saved notebook (outputs are stored inside the file) and the conclusions are written from the numbers in it.

In [ ]:
# What: saves every reported line to results/summary.txt for the README.
# Expected output: one confirmation line.
with open(os.path.join(RESULTS_FOLDER, "summary.txt"), "w", encoding="utf-8") as file:
    file.write("\n".join(summary_lines) + "\n")
print("Saved", len(summary_lines), "lines to", os.path.join(RESULTS_FOLDER, "summary.txt"))